| **ID**| **Name**| 
|-|-|
| 22280075     | Huynh Thao Quynh  | 
| 22280091 | Nguyen Ngoc Thanh Thu  | 

# **Final Project**

## **Problem stament :**     

The widespread dissemination of fake news and propaganda presents serious societal risks, including the erosion of public trust, political polarization, manipulation of elections, and the spread of harmful misinformation during crises such as pandemics or conflicts. From an NLP perspective, detecting fake news is fraught with challenges. Linguistically, fake news often mimics the tone and structure of legitimate journalism, making it difficult to distinguish using surface-level features. The absence of reliable and up-to-date labeled datasets, especially across multiple languages and regions, hampers the effectiveness of supervised learning models. Additionally, the dynamic and adversarial nature of misinformation means that malicious actors constantly evolve their language and strategies to bypass detection systems. Cultural context, sarcasm, satire, and implicit bias further complicate automated analysis. Moreover, NLP models risk amplifying biases present in training data, leading to unfair classifications and potential censorship of legitimate content. These challenges underscore the need for cautious, context-aware approaches, as the failure to address them can inadvertently contribute to misinformation, rather than mitigate it.



Use datasets in link : https://drive.google.com/drive/folders/1mrX3vPKhEzxG96OCPpCeh9F8m_QKCM4z?usp=sharing
to complete requirement.

## **About dataset:**

* **True Articles**:

  * **File**: `MisinfoSuperset_TRUE.csv`
  * **Sources**:

    * Reputable media outlets like **Reuters**, **The New York Times**, **The Washington Post**, etc.

* **Fake/Misinformation/Propaganda Articles**:

  * **File**: `MisinfoSuperset_FAKE.csv`
  * **Sources**:

    * **American right-wing extremist websites** (e.g., Redflag Newsdesk, Breitbart, Truth Broadcast Network)
    * **Public dataset** from:

      * Ahmed, H., Traore, I., & Saad, S. (2017): "Detection of Online Fake News Using N-Gram Analysis and Machine Learning Techniques" *(Springer LNCS 10618)*



## **Requirement**

A team consisting of three members must complete a project that involves applying the methods learned from the beginning of the course up to the present. The team is expected to follow and document the entire machine learning workflow, which includes the following steps:

1. **Data Preprocessing**: Clean and prepare the dataset,etc.

2. **Exploratory Data Analysis (EDA)**: Explore and visualize the data.

3. **Model Building**: Select and build one or more machine learning models suitable for the problem at hand.

4. **Hyperparameter set up**: Set and adjust the model's hyperparameters using appropriate methods to improve performance.

5. **Model Training**: Train the model(s) on the training dataset.

6. **Performance Evaluation**: Evaluate the trained model(s) using appropriate metrics (e.g., accuracy, precision, recall, F1-score, confusion matrix, etc.) and validate their performance on unseen data.

7. **Conclusion**: Summarize the results, discuss the model's strengths and weaknesses, and suggest possible improvements or future work.





# MY PIPELINE

## **01. Import Libraries and Load Data**

In [1]:
# Import Libararies
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from wordcloud import WordCloud
from nltk.sentiment import SentimentIntensityAnalyzer
import plotly.graph_objects as go
from collections import Counter
from textblob import TextBlob
# Vectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import VotingClassifier
# Clasifiers
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
#Train Test split
from sklearn.model_selection import train_test_split
#pipeline
from sklearn.pipeline import Pipeline
# for NLP
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from bs4 import BeautifulSoup
from sklearn.utils import shuffle
import torch
from transformers import Trainer
# MultinomialNB
from sklearn.naive_bayes import MultinomialNB
# import SVC
from sklearn.svm import SVC
from scipy.special import softmax
import joblib  # To save and load models
# pipeline
from imblearn.pipeline import Pipeline
from sklearn.pipeline import make_pipeline
# Count Vectorizer 
from sklearn.base import TransformerMixin
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from torch.utils.data import Dataset
import plotly.io as pio
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [2]:
fake_data = pd.read_csv('dataset_misinfo/DataSet_Misinfo_FAKE.csv')
real_data = pd.read_csv('dataset_misinfo/DataSet_Misinfo_TRUE.csv')

fake_data = fake_data.drop(columns=['Unnamed: 0'])
real_data = real_data.drop(columns=['Unnamed: 0'])
fake_data['label'] = 0
real_data['label'] = 1

df = pd.concat([real_data, fake_data], axis=0)

## **02. EDA**

In [3]:
# Thông tin kích thước và cấu trúc
print("Dataset shape:", df.shape)

# Số lượng mỗi nhãn
print(df['label'].value_counts())

# Kiểm tra và xử lý missing values
missing = df.isnull().sum()
print("Missing values:\n", missing)

if missing.sum() > 0:
    df = df.dropna().reset_index(drop=True)

Dataset shape: (78617, 2)
label
0    43642
1    34975
Name: count, dtype: int64
Missing values:
 text     29
label     0
dtype: int64


### **Visualize**

In [4]:
def plot_word_cloud(data, title):
    text_data = ' '.join(data.astype(str))
    if not text_data.strip():  
        print(f"No text data available for {title}")
        return
    
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text_data)
    
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title(title)
    plt.show()

# Generate word clouds for Real (label=1) and Fake (label=0) news
plot_word_cloud(df[df['label'] == 1]['text'], 'Word Cloud for Real News')
plot_word_cloud(df[df['label'] == 0]['text'], 'Word Cloud for Fake News')

KeyboardInterrupt: 

In [ ]:
df['article_length'] = df['text'].apply(lambda x: len(x.split()))
df.head()

In [ ]:
import plotly.express as px

In [ ]:
# Count the number of articles by label
article_count = df['label'].value_counts().reset_index()
article_count.columns = ['label', 'count']

# Map labels 0, 1 to Fake, Real
article_count['label'] = article_count['label'].map({0: 'Fake', 1: 'Real'})

print("Number of articles by label:")
print(article_count)

# Create a bar chart 
fig = go.Figure()

fig.add_trace(go.Bar(
    x=article_count['label'],
    y=article_count['count'],
    marker_color=['#FF6B6B', '#4ECDC4'],  
    text=article_count['count'], 
    textposition='outside',
    texttemplate='%{text:.0f}',  
    width=0.3,  
    marker_line=dict(color='#000000', width=1)  
))

fig.update_layout(
    title=dict(
        text='Number of Articles by News Type',
        font=dict(size=20, color='#333333'),
        x=0.5,
        xanchor='center'
    ),
    xaxis_title='News Type',
    yaxis_title='Number of Articles',
    font=dict(family='Arial', size=14, color='#333333'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(
        showgrid=False,
        title_font=dict(size=16),
        tickfont=dict(size=14)
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='lightgrey',
        gridwidth=1,
        title_font=dict(size=16),
        tickfont=dict(size=14),
        range=[0, max(article_count['count']) * 1.2] 
    ),
    bargap=0.05,  
    showlegend=False,
    margin=dict(l=50, r=50, t=80, b=50),
    height=500
)

fig.show()

In [ ]:
# Calculate the average article length by label
article_length_avg = df.groupby('label')['article_length'].mean().reset_index()
article_length_avg.columns = ['label', 'avg_length']

# Map labels 0, 1 to Fake, Real
article_length_avg['label'] = article_length_avg['label'].map({0: 'Fake', 1: 'Real'})

print("Average article length by label:")
print(article_length_avg)

# Create a bar chart
fig = go.Figure()

fig.add_trace(go.Bar(
    x=article_length_avg['label'],
    y=article_length_avg['avg_length'],
    marker_color=['#FF6B6B', '#4ECDC4'],  
    text=article_length_avg['avg_length'], 
    textposition='outside',
    texttemplate='%{text:.1f}', 
    width=0.3, 
    marker_line=dict(color='#000000', width=1)  
))

fig.update_layout(
    title=dict(
        text='Average Article Length by News Type',
        font=dict(size=20, color='#333333'),
        x=0.5,
        xanchor='center'
    ),
    xaxis_title='News Type',
    yaxis_title='Average Length (words)',
    font=dict(family='Arial', size=14, color='#333333'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(
        showgrid=False,
        title_font=dict(size=16),
        tickfont=dict(size=14)
    ),
    yaxis=dict(
        showgrid=True,
        gridcolor='lightgrey',
        gridwidth=1,
        title_font=dict(size=16),
        tickfont=dict(size=14),
        range=[0, max(article_length_avg['avg_length']) * 1.2]  
    ),
    bargap=0.05,  
    showlegend=False,
    margin=dict(l=50, r=50, t=80, b=50),
    height=500
)

fig.show()

In [ ]:
from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import TextBlob

nltk.download('vader_lexicon')
sia = SentimentIntensityAnalyzer()

In [ ]:
def get_sentiment(text):
    return sia.polarity_scores(text)['compound']

df['sentiment'] = df['text'].apply(get_sentiment)

In [ ]:
# Calculate mean, max, and min sentiment scores for each label
mean_sentiment = df.groupby('label')['sentiment'].mean().reset_index()
max_sentiment = df.groupby('label')['sentiment'].max().reset_index()
min_sentiment = df.groupby('label')['sentiment'].min().reset_index()

summary_sentiment = pd.merge(mean_sentiment, max_sentiment, on='label', suffixes=('_mean', '_max'))
summary_sentiment = pd.merge(summary_sentiment, min_sentiment, on='label')
summary_sentiment.rename(columns={'sentiment': 'sentiment_min'}, inplace=True)

for index, row in summary_sentiment.iterrows():
    print(f"Label: {row['label']}, Mean Sentiment: {row['sentiment_mean']:.2f}, Max Sentiment: {row['sentiment_max']:.2f}, Min Sentiment: {row['sentiment_min']:.2f}")

# Create a violin plot for sentiment scores
fig = px.violin(df, 
                 x='label', 
                 y='sentiment', 
                 box=True, 
                 points='all', 
                 title='Sentiment Polarity Distribution by Label',
                 color='label',  # Color by label
                 color_discrete_sequence=['#FF6B6B', '#4ECDC4'])  # Custom colors

fig.update_traces(meanline_visible=True)  
fig.update_layout(yaxis_title='Sentiment Score',
                  xaxis_title='News Category',
                  legend_title='Label',
                  legend=dict(x=0.85, y=0.9))  

fig.show()

In [ ]:
def get_textblob_sentiment(text):
    blob = TextBlob(str(text))
    return blob.sentiment.polarity, blob.sentiment.subjectivity

df['textblob_sentiment'], df['textblob_subjectivity'] = zip(*df['text'].apply(get_textblob_sentiment))

In [ ]:
# Calculate mean, max, and min sentiment scores for each label
mean_sentiment = df.groupby('label')['textblob_sentiment'].mean().reset_index()
max_sentiment = df.groupby('label')['textblob_sentiment'].max().reset_index()
min_sentiment = df.groupby('label')['textblob_sentiment'].min().reset_index()

summary_sentiment = pd.merge(mean_sentiment, max_sentiment, on='label', suffixes=('_mean', '_max'))
summary_sentiment = pd.merge(summary_sentiment, min_sentiment, on='label')
summary_sentiment.rename(columns={'textblob_sentiment': 'textblob_sentiment_min'}, inplace=True)

for index, row in summary_sentiment.iterrows():
    print(f"Label: {row['label']}, Mean Sentiment: {row['textblob_sentiment_mean']:.2f}, Max Sentiment: {row['textblob_sentiment_max']:.2f}, Min Sentiment: {row['textblob_sentiment_min']:.2f}")

# Create the violin plot
fig = px.violin(df, x='label', y='textblob_sentiment', box=True, points='all',
                 title='TextBlob Sentiment Polarity Distribution by Label',
                 color='label',  
                 color_discrete_sequence=px.colors.qualitative.Set2)

for index, row in summary_sentiment.iterrows():
    # Annotate mean score
    fig.add_annotation(
        x=row['label'],
        y=row['textblob_sentiment_mean'],
        text=f'Mean: {row["textblob_sentiment_mean"]:.2f}',  
        showarrow=True,
        arrowhead=2,
        ax=0,
        ay=-40,  
        font=dict(color='black', size=12)
    )

    fig.add_annotation(
        x=row['label'],
        y=row['textblob_sentiment_max'],
        text=f'Max: {row["textblob_sentiment_max"]:.2f}',  
        showarrow=True,
        arrowhead=2,
        ax=0,
        ay=-20,
        font=dict(color='red', size=12) 
    )
    # Annotate min score
    fig.add_annotation(
        x=row['label'],
        y=row['textblob_sentiment_min'],
        text=f'Min: {row["textblob_sentiment_min"]:.2f}',  
        showarrow=True,
        arrowhead=2,
        ax=0,
        ay=0, 
        font=dict(color='blue', size=12) 
    )

fig.update_layout(
    legend_title_text='Label',
    yaxis_title='TextBlob Sentiment Polarity',
    xaxis_title='Label',
    template='plotly_white'  
)

fig.show()

In [ ]:
# Define stopwords
stop_words = set(stopwords.words('english'))

# Function to get top N words, excluding stopwords
def get_top_n_words(corpus, n=10):
    words = ' '.join(corpus).split()
    words = [word for word in words if word.isalpha() and word.lower() not in stop_words]
    common_words = Counter(words)
    return common_words.most_common(n)

# Function to plot top N words 
def plot_top_n_words(data, title, n=10, base_color='#000000', highlight_color='#000000'):
    top_words = get_top_n_words(data['text'], n)
    if not top_words:  
        print(f"No significant words to plot for {title}")
        return
    
    words, counts = zip(*top_words)
    
    counts = list(counts)
    words = list(words)

    print(f"Top words for {title}:")
    for word, count in zip(words, counts):
        print(f"{word}: {count}")
    
    max_count_index = counts.index(max(counts))

    colors = [highlight_color if i == max_count_index else base_color for i in range(len(counts))]

    fig = px.bar(x=words, y=counts, 
                 title=title, 
                 labels={'x': 'Words', 'y': 'Counts'},
                 color=words,  
                 color_discrete_map={word: color for word, color in zip(words, colors)},
                 text=counts, 
                 text_auto=True)  

    fig.update_traces(textposition='outside')
    
    fig.update_layout(
        title=dict(text=title, 
                   font=dict(size=20, color='#333333'), 
                   x=0.5, 
                   xanchor='center'),
        xaxis_title='Words',
        yaxis_title='Counts',
        font=dict(family='Arial', size=14, color='#333333'),
        plot_bgcolor='white',
        paper_bgcolor='white',
        xaxis=dict(
            showgrid=False, 
            title_font=dict(size=16),
            tickfont=dict(size=14),
            tickangle=45  
        ),
        yaxis=dict(
            showgrid=True, 
            gridcolor='lightgrey', 
            gridwidth=1,
            title_font=dict(size=16),
            tickfont=dict(size=14)
        ),
        showlegend=False,
        margin=dict(l=50, r=50, t=80, b=50),
        height=500
    )
    
    fig.show()

real_base_color = '#4ECDC4'  
real_highlight_color = '#2A7C73' 
fake_base_color = '#FF6B6B'  
fake_highlight_color = '#B53030'

In [ ]:
# Plot top 10 words for Real and Fake news 
plot_top_n_words(df[df['label'] == 1], 'Top 10 Words in Real News', n=10, 
                 base_color=real_base_color, highlight_color=real_highlight_color)
plot_top_n_words(df[df['label'] == 0], 'Top 10 Words in Fake News', n=10, 
                 base_color=fake_base_color, highlight_color=fake_highlight_color)

In [ ]:
# Count the number of real and fake news articles
label_counts = df['label'].value_counts()

label_df = pd.DataFrame({'label': label_counts.index, 'count': label_counts.values})
label_df['label'] = label_df['label'].map({0: 'Fake', 1: 'Real'})  

fig = make_subplots(
    rows=1, 
    cols=2, 
    specs=[[{'type': 'pie'}, {'type': 'bar'}]],
    subplot_titles=("Pie Chart of News Labels", "Bar Plot of News Labels"),
    horizontal_spacing=0.15 
)

# Pie Chart
pie_chart = go.Pie(
    labels=label_df['label'], 
    values=label_df['count'], 
    hole=0.4,
    marker_colors=['#FF6B6B', '#4ECDC4'], 
    textinfo='percent+label',
    hoverinfo='label+percent+value'
)
fig.add_trace(pie_chart, row=1, col=1)

# Bar Plot
bar_plot = go.Bar(
    x=label_df['label'], 
    y=label_df['count'], 
    marker_color=['#FF6B6B', '#4ECDC4'],
    text=label_df['count'],
    textposition='outside'
)
fig.add_trace(bar_plot, row=1, col=2)

fig.update_layout(
    title=dict(
        text='Distribution of Real and Fake News',
        font=dict(size=20, color='#333333'),
        x=0.5,
        xanchor='center'
    ),
    font=dict(family='Arial', size=14, color='#333333'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=True,
    legend=dict(
        title='News Type',
        x=0.85,
        y=0.9
    ),
    height=500,
    width=1000,  
    margin=dict(l=50, r=50, t=80, b=50)
)

fig.update_yaxes(
    title_text='Number of Articles',
    showgrid=True,
    gridcolor='lightgrey',
    gridwidth=1,
    tickformat=',',
    title_font=dict(size=16),
    tickfont=dict(size=14),
    range=[0, max(label_df['count']) * 1.2],
    row=1, col=2
)
fig.update_xaxes(
    title_text='News Type',
    title_font=dict(size=16),
    tickfont=dict(size=14),
    row=1, col=2
)

fig.show()

## **03. Preprocessing Data**

In [ ]:
def check_html_tags(text):
    html_pattern = r'<[^>]+>'
    return bool(re.search(html_pattern, text))
print(f"Số dòng còn chứa thẻ HTML: {df['text'].apply(check_html_tags).sum()}")

In [ ]:
def clean_text_for_transformer(text):
    # Remove escape characters
    text = re.sub(r'[\t\r\n]', ' ', text)

    # Check if the text looks like HTML before using BeautifulSoup
    if re.search(r'<[^>]+>', text):
        text = BeautifulSoup(text, "html.parser").get_text()

    # Remove URLs
    text = re.sub(r'(https?://)([^/\s]+)([^\s]*)', r'\2', text)

    # Remove repeated special chars (__, --, .., etc.)
    text = re.sub(r'[_~+\-]{2,}', ' ', text)

    # Normalize multiple spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [ ]:
# Load the model with the full name and disable components for speed
nlp = spacy.load('en_core_web_sm', disable=['ner', 'parser'])

def clean_text_for_ML(text):
    # Convert to lowercase
    text = str(text).lower()
    
    # Check if the text looks like HTML before using BeautifulSoup
    if re.search(r'<[^>]+>', text):
        text = BeautifulSoup(text, "html.parser").get_text()

    # Remove URLs
    text = re.sub(r'(https?://)([^/\s]+)([^\s]*)', r'\2', text)

    # Remove repeated special chars (__, --, .., etc.)
    text = re.sub(r'[_~+\-]{2,}', ' ', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Normalize multiple spaces
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

def lemmatize_texts(texts):
    num_proc = max(1, multiprocessing.cpu_count() // 2)

    # Dùng spaCy để lemmatize và loại stopwords
    docs = nlp.pipe(texts, batch_size=1000, n_process=num_proc)
    return [' '.join([token.lemma_ for token in doc if not token.is_stop]) for doc in docs]


In [ ]:
df['clean_text_trans'] = df['text'].apply(clean_text_for_transformer)
df['clean_text_ml'] = df['text'].apply(clean_text_for_ML)
df['clean_text_ml'] = lemmatize_texts(df['clean_text_ml'].tolist())